In [24]:
# [TOP CELL] Choose input JSON + quick unit diagnostics
import json
from pathlib import Path
from collections import Counter
import pandas as pd

# 1) Pick which JSON to use
INPUT_JSON = Path('output_filled_name_unit_removed.json')
# Examples:
# INPUT_JSON = Path('output_filled_unit_size_fixed.json')
# INPUT_JSON = Path('output_filled_unit_singleword_only.json')
# INPUT_JSON = Path('output_filled_name_unit_removed.json')

if not INPUT_JSON.exists():
    raise FileNotFoundError(f'Input JSON not found: {INPUT_JSON.resolve()}')

# 2) Load
with INPUT_JSON.open('r', encoding='utf-8') as f:
    data = json.load(f)

# 3) Unit stats
unit_counter = Counter()
empty_unit_count = 0
for item in data:
    u = str(item.get('unit', '') or '').strip().lower()
    if u:
        unit_counter[u] += 1
    else:
        empty_unit_count += 1

total_rows = len(data)
rows_with_unit = total_rows - empty_unit_count
unique_units = len(unit_counter)

print(f'Input JSON: {INPUT_JSON.resolve()}')
print(f'Total rows: {total_rows}')
print(f'Rows with non-empty unit: {rows_with_unit}')
print(f'Rows with empty unit: {empty_unit_count}')
print(f'Unique unit types: {unique_units}')

unit_df = pd.DataFrame(unit_counter.items(), columns=['unit', 'count']).sort_values(
    by=['count', 'unit'], ascending=[False, True]
).reset_index(drop=True)

UNIT_COUNTS_CSV = Path('unit_counts_selected_input.csv')
TOP30_UNITS_CSV = Path('unit_top30_selected_input.csv')
SUMMARY_CSV = Path('unit_summary_selected_input.csv')

summary_df = pd.DataFrame([
    {'metric': 'input_json', 'value': str(INPUT_JSON)},
    {'metric': 'total_rows', 'value': total_rows},
    {'metric': 'rows_with_non_empty_unit', 'value': rows_with_unit},
    {'metric': 'rows_with_empty_unit', 'value': empty_unit_count},
    {'metric': 'unique_unit_types', 'value': unique_units},
])

unit_df.to_csv(UNIT_COUNTS_CSV, index=False, encoding='utf-8')
unit_df.head(30).to_csv(TOP30_UNITS_CSV, index=False, encoding='utf-8')
summary_df.to_csv(SUMMARY_CSV, index=False, encoding='utf-8')

print('Saved files:')
print(f'- {UNIT_COUNTS_CSV.resolve()}')
print(f'- {TOP30_UNITS_CSV.resolve()}')
print(f'- {SUMMARY_CSV.resolve()}')

Input JSON: /Users/kyungmin/Desktop/Projects/food_chart_public/food_model/gptgram_model/scrape/output_filled_name_unit_removed.json
Total rows: 18222
Rows with non-empty unit: 14157
Rows with empty unit: 4065
Unique unit types: 63
Saved files:
- /Users/kyungmin/Desktop/Projects/food_chart_public/food_model/gptgram_model/scrape/unit_counts_selected_input.csv
- /Users/kyungmin/Desktop/Projects/food_chart_public/food_model/gptgram_model/scrape/unit_top30_selected_input.csv
- /Users/kyungmin/Desktop/Projects/food_chart_public/food_model/gptgram_model/scrape/unit_summary_selected_input.csv


In [23]:
# MERGED PIPELINE CELL (code)
# combines old steps 1, 3, 6, 7, 8

import json
import re
from pathlib import Path
from collections import Counter
import pandas as pd

# 0) Input/Output paths
SOURCE_JSON = Path('output_filled_copy.json')
STEP1_JSON = Path('output_filled_unit_size_fixed.json')
STEP2_JSON = Path('output_filled_unit_singleword_only.json')
STEP3_MATCH_JSON = Path('name_contains_its_own_unit_rows.json')
FINAL_JSON = Path('output_filled_name_unit_removed.json')

UNIT_COUNTS_AFTER_STEP2_CSV = Path('unit_counts_singleword_only.csv')
MULTIWORD_REMOVED_CSV = Path('removed_multiword_units.csv')
MULTIWORD_SUMMARY_CSV = Path('multiword_unit_cleanup_summary.csv')
OWN_UNIT_MATCH_CSV = Path('name_contains_its_own_unit_rows.csv')
NAME_UNIT_REMOVED_ROWS_CSV = Path('name_unit_removed_rows.csv')
NAME_UNIT_REMOVED_SUMMARY_CSV = Path('name_unit_removed_summary.csv')

if not SOURCE_JSON.exists():
    raise FileNotFoundError(f'Source JSON not found: {SOURCE_JSON.resolve()}')

with SOURCE_JSON.open('r', encoding='utf-8') as f:
    rows = json.load(f)

# STEP 1) remove small/medium/large from unit and move to size if needed
SIZE_REGEX = re.compile(r'\b(small|medium|large)\b', re.IGNORECASE)
rows_step1 = []
size_word_removed = 0
size_moved = 0

for item in rows:
    rec = dict(item)
    original_unit = str(rec.get('unit', '') or '')
    original_size = str(rec.get('size', '') or '').strip()

    matches = SIZE_REGEX.findall(original_unit)
    if matches:
        cleaned_unit = SIZE_REGEX.sub('', original_unit)
        cleaned_unit = re.sub(r'\s+', ' ', cleaned_unit).strip()

        if cleaned_unit != original_unit.strip():
            size_word_removed += 1
        rec['unit'] = cleaned_unit

        if not original_size:
            rec['size'] = matches[0].lower()
            size_moved += 1

    rows_step1.append(rec)

with STEP1_JSON.open('w', encoding='utf-8') as f:
    json.dump(rows_step1, f, ensure_ascii=False, indent=2)

# STEP 2) normalize thin/thick slice -> slice, then remove multiword units
UNIT_NORMALIZE_MAP = {
    'thin slice': 'slice',
    'thick slice': 'slice',
}

def normalize_unit(u):
    normalized = ' '.join(str(u or '').strip().lower().split())
    mapped = UNIT_NORMALIZE_MAP.get(normalized, normalized)
    return mapped, mapped != normalized

def unit_word_count(u):
    u = str(u or '').strip()
    if not u:
        return 0
    return len(' '.join(u.split()).split(' '))

rows_step2 = []
removed_multiword = []
normalized_to_slice = 0

for rec in rows_step1:
    item = dict(rec)
    unit_norm, changed = normalize_unit(item.get('unit', ''))
    item['unit'] = unit_norm
    if changed:
        normalized_to_slice += 1

    if unit_word_count(unit_norm) >= 2:
        removed_multiword.append({
            'unit': unit_norm,
            'size': item.get('size', ''),
            'name': item.get('name', ''),
            'gram': item.get('gram', ''),
            'word_count': unit_word_count(unit_norm),
        })
    else:
        rows_step2.append(item)

with STEP2_JSON.open('w', encoding='utf-8') as f:
    json.dump(rows_step2, f, ensure_ascii=False, indent=2)

removed_df = pd.DataFrame(removed_multiword)
removed_df.to_csv(MULTIWORD_REMOVED_CSV, index=False, encoding='utf-8')

unit_counter_step2 = Counter(str(r.get('unit', '') or '').strip().lower() for r in rows_step2)
unit_counts_df = pd.DataFrame(unit_counter_step2.items(), columns=['unit', 'count']).sort_values(
    by=['count', 'unit'], ascending=[False, True]
).reset_index(drop=True)
unit_counts_df.to_csv(UNIT_COUNTS_AFTER_STEP2_CSV, index=False, encoding='utf-8')

pd.DataFrame([
    {'metric': 'rows_before_step2', 'value': len(rows_step1)},
    {'metric': 'rows_unit_normalized_to_slice', 'value': normalized_to_slice},
    {'metric': 'rows_removed_multiword_unit', 'value': len(removed_multiword)},
    {'metric': 'rows_after_step2', 'value': len(rows_step2)},
]).to_csv(MULTIWORD_SUMMARY_CSV, index=False, encoding='utf-8')

# STEP 3) find rows where own unit appears in name
matched_rows = []
for i, r in enumerate(rows_step2):
    unit = str(r.get('unit', '') or '').strip().lower()
    name = str(r.get('name', '') or '').strip().lower()
    if not unit or not name:
        continue
    p = re.compile(r'\b' + re.escape(unit) + r'\b', re.IGNORECASE)
    if p.search(name):
        matched_rows.append({
            'row_index': i,
            'unit': r.get('unit', ''),
            'size': r.get('size', ''),
            'name': r.get('name', ''),
            'gram': r.get('gram', ''),
        })

with STEP3_MATCH_JSON.open('w', encoding='utf-8') as f:
    json.dump(matched_rows, f, ensure_ascii=False, indent=2)

pd.DataFrame(matched_rows).to_csv(OWN_UNIT_MATCH_CSV, index=False, encoding='utf-8')

# STEP 4) remove own unit token from name for matched rows
matched_idx = {int(r['row_index']) for r in matched_rows}
changed_rows = []
final_rows = []

for i, r in enumerate(rows_step2):
    item = dict(r)
    if i in matched_idx:
        unit = str(item.get('unit', '') or '').strip()
        name = str(item.get('name', '') or '').strip()
        if unit and name:
            p = re.compile(r'\b' + re.escape(unit) + r'\b', re.IGNORECASE)
            cleaned_name = p.sub('', name)
            cleaned_name = re.sub(r'\s+', ' ', cleaned_name).strip(' ,.-')
            if cleaned_name != name:
                changed_rows.append({
                    'row_index': i,
                    'unit': unit,
                    'before_name': name,
                    'after_name': cleaned_name,
                    'size': item.get('size', ''),
                    'gram': item.get('gram', ''),
                })
                item['name'] = cleaned_name
    final_rows.append(item)

with FINAL_JSON.open('w', encoding='utf-8') as f:
    json.dump(final_rows, f, ensure_ascii=False, indent=2)

pd.DataFrame(changed_rows).to_csv(NAME_UNIT_REMOVED_ROWS_CSV, index=False, encoding='utf-8')
pd.DataFrame([
    {'metric': 'input_rows_step4', 'value': len(rows_step2)},
    {'metric': 'matched_rows_loaded', 'value': len(matched_rows)},
    {'metric': 'names_changed', 'value': len(changed_rows)},
]).to_csv(NAME_UNIT_REMOVED_SUMMARY_CSV, index=False, encoding='utf-8')

print('Done merged cleanup pipeline')
print(f'Source rows: {len(rows)}')
print(f'STEP1 saved: {STEP1_JSON.resolve()} | removed_size_words={size_word_removed} | moved_to_size={size_moved}')
print(f'STEP2 saved: {STEP2_JSON.resolve()} | normalized_to_slice={normalized_to_slice} | removed_multiword={len(removed_multiword)}')
print(f'STEP3 match rows: {len(matched_rows)} | file={STEP3_MATCH_JSON.resolve()}')
print(f'STEP4 final rows: {len(final_rows)} | names_changed={len(changed_rows)} | file={FINAL_JSON.resolve()}')
print('CSV outputs:')
print(f'- {UNIT_COUNTS_AFTER_STEP2_CSV.resolve()}')
print(f'- {MULTIWORD_REMOVED_CSV.resolve()}')
print(f'- {MULTIWORD_SUMMARY_CSV.resolve()}')
print(f'- {OWN_UNIT_MATCH_CSV.resolve()}')
print(f'- {NAME_UNIT_REMOVED_ROWS_CSV.resolve()}')
print(f'- {NAME_UNIT_REMOVED_SUMMARY_CSV.resolve()}')

Done merged cleanup pipeline
Source rows: 18352
STEP1 saved: /Users/kyungmin/Desktop/Projects/food_chart_public/food_model/gptgram_model/scrape/output_filled_unit_size_fixed.json | removed_size_words=155 | moved_to_size=155
STEP2 saved: /Users/kyungmin/Desktop/Projects/food_chart_public/food_model/gptgram_model/scrape/output_filled_unit_singleword_only.json | normalized_to_slice=102 | removed_multiword=130
STEP3 match rows: 34 | file=/Users/kyungmin/Desktop/Projects/food_chart_public/food_model/gptgram_model/scrape/name_contains_its_own_unit_rows.json
STEP4 final rows: 18222 | names_changed=34 | file=/Users/kyungmin/Desktop/Projects/food_chart_public/food_model/gptgram_model/scrape/output_filled_name_unit_removed.json
CSV outputs:
- /Users/kyungmin/Desktop/Projects/food_chart_public/food_model/gptgram_model/scrape/unit_counts_singleword_only.csv
- /Users/kyungmin/Desktop/Projects/food_chart_public/food_model/gptgram_model/scrape/removed_multiword_units.csv
- /Users/kyungmin/Desktop/Pro

# MERGED PIPELINE CELL (combines old steps 1, 3, 6, 7, 8)
# Keeps one end-to-end cleanup flow from source JSON.

import json
import re
from pathlib import Path
from collections import Counter
import pandas as pd

# 0) Input/Output paths
SOURCE_JSON = Path('output_filled_copy.json')
STEP1_JSON = Path('output_filled_unit_size_fixed.json')
STEP2_JSON = Path('output_filled_unit_singleword_only.json')
STEP3_MATCH_JSON = Path('name_contains_its_own_unit_rows.json')
FINAL_JSON = Path('output_filled_name_unit_removed.json')

UNIT_COUNTS_AFTER_STEP2_CSV = Path('unit_counts_singleword_only.csv')
MULTIWORD_REMOVED_CSV = Path('removed_multiword_units.csv')
MULTIWORD_SUMMARY_CSV = Path('multiword_unit_cleanup_summary.csv')
OWN_UNIT_MATCH_CSV = Path('name_contains_its_own_unit_rows.csv')
NAME_UNIT_REMOVED_ROWS_CSV = Path('name_unit_removed_rows.csv')
NAME_UNIT_REMOVED_SUMMARY_CSV = Path('name_unit_removed_summary.csv')

if not SOURCE_JSON.exists():
    raise FileNotFoundError(f'Source JSON not found: {SOURCE_JSON.resolve()}')

with SOURCE_JSON.open('r', encoding='utf-8') as f:
    rows = json.load(f)

# ------------------------------------------------------------------
# STEP 1) unit에서 small/medium/large 제거 + size로 이동
# ------------------------------------------------------------------
SIZE_REGEX = re.compile(r'\b(small|medium|large)\b', re.IGNORECASE)
rows_step1 = []
size_word_removed = 0
size_moved = 0

for item in rows:
    rec = dict(item)
    original_unit = str(rec.get('unit', '') or '')
    original_size = str(rec.get('size', '') or '').strip()

    matches = SIZE_REGEX.findall(original_unit)
    if matches:
        cleaned_unit = SIZE_REGEX.sub('', original_unit)
        cleaned_unit = re.sub(r'\s+', ' ', cleaned_unit).strip()

        if cleaned_unit != original_unit.strip():
            size_word_removed += 1
        rec['unit'] = cleaned_unit

        if not original_size:
            rec['size'] = matches[0].lower()
            size_moved += 1

    rows_step1.append(rec)

with STEP1_JSON.open('w', encoding='utf-8') as f:
    json.dump(rows_step1, f, ensure_ascii=False, indent=2)

# ------------------------------------------------------------------
# STEP 2) thin/thick slice -> slice 정규화 후, 다단어 unit 제거
# ------------------------------------------------------------------
UNIT_NORMALIZE_MAP = {
    'thin slice': 'slice',
    'thick slice': 'slice',
}

def normalize_unit(u):
    normalized = ' '.join(str(u or '').strip().lower().split())
    mapped = UNIT_NORMALIZE_MAP.get(normalized, normalized)
    return mapped, mapped != normalized

def unit_word_count(u):
    u = str(u or '').strip()
    if not u:
        return 0
    return len(' '.join(u.split()).split(' '))

rows_step2 = []
removed_multiword = []
normalized_to_slice = 0

for rec in rows_step1:
    item = dict(rec)
    unit_norm, changed = normalize_unit(item.get('unit', ''))
    item['unit'] = unit_norm
    if changed:
        normalized_to_slice += 1

    if unit_word_count(unit_norm) >= 2:
        removed_multiword.append({
            'unit': unit_norm,
            'size': item.get('size', ''),
            'name': item.get('name', ''),
            'gram': item.get('gram', ''),
            'word_count': unit_word_count(unit_norm),
        })
    else:
        rows_step2.append(item)

with STEP2_JSON.open('w', encoding='utf-8') as f:
    json.dump(rows_step2, f, ensure_ascii=False, indent=2)

removed_df = pd.DataFrame(removed_multiword)
removed_df.to_csv(MULTIWORD_REMOVED_CSV, index=False, encoding='utf-8')

unit_counter_step2 = Counter(str(r.get('unit', '') or '').strip().lower() for r in rows_step2)
unit_counts_df = pd.DataFrame(unit_counter_step2.items(), columns=['unit', 'count']).sort_values(
    by=['count', 'unit'], ascending=[False, True]
).reset_index(drop=True)
unit_counts_df.to_csv(UNIT_COUNTS_AFTER_STEP2_CSV, index=False, encoding='utf-8')

pd.DataFrame([
    {'metric': 'rows_before_step2', 'value': len(rows_step1)},
    {'metric': 'rows_unit_normalized_to_slice', 'value': normalized_to_slice},
    {'metric': 'rows_removed_multiword_unit', 'value': len(removed_multiword)},
    {'metric': 'rows_after_step2', 'value': len(rows_step2)},
]).to_csv(MULTIWORD_SUMMARY_CSV, index=False, encoding='utf-8')

# ------------------------------------------------------------------
# STEP 3) 각 row에서 own unit이 name에 포함된 행 찾기
# ------------------------------------------------------------------
matched_rows = []
for i, r in enumerate(rows_step2):
    unit = str(r.get('unit', '') or '').strip().lower()
    name = str(r.get('name', '') or '').strip().lower()
    if not unit or not name:
        continue
    p = re.compile(r'\b' + re.escape(unit) + r'\b', re.IGNORECASE)
    if p.search(name):
        matched_rows.append({
            'row_index': i,
            'unit': r.get('unit', ''),
            'size': r.get('size', ''),
            'name': r.get('name', ''),
            'gram': r.get('gram', ''),
        })

with STEP3_MATCH_JSON.open('w', encoding='utf-8') as f:
    json.dump(matched_rows, f, ensure_ascii=False, indent=2)

pd.DataFrame(matched_rows).to_csv(OWN_UNIT_MATCH_CSV, index=False, encoding='utf-8')

# ------------------------------------------------------------------
# STEP 4) STEP3에서 잡힌 row의 name에서 own unit 제거
# ------------------------------------------------------------------
matched_idx = {int(r['row_index']) for r in matched_rows}
changed_rows = []
final_rows = []

for i, r in enumerate(rows_step2):
    item = dict(r)
    if i in matched_idx:
        unit = str(item.get('unit', '') or '').strip()
        name = str(item.get('name', '') or '').strip()
        if unit and name:
            p = re.compile(r'\b' + re.escape(unit) + r'\b', re.IGNORECASE)
            cleaned_name = p.sub('', name)
            cleaned_name = re.sub(r'\s+', ' ', cleaned_name).strip(' ,.-')
            if cleaned_name != name:
                changed_rows.append({
                    'row_index': i,
                    'unit': unit,
                    'before_name': name,
                    'after_name': cleaned_name,
                    'size': item.get('size', ''),
                    'gram': item.get('gram', ''),
                })
                item['name'] = cleaned_name
    final_rows.append(item)

with FINAL_JSON.open('w', encoding='utf-8') as f:
    json.dump(final_rows, f, ensure_ascii=False, indent=2)

pd.DataFrame(changed_rows).to_csv(NAME_UNIT_REMOVED_ROWS_CSV, index=False, encoding='utf-8')
pd.DataFrame([
    {'metric': 'input_rows_step4', 'value': len(rows_step2)},
    {'metric': 'matched_rows_loaded', 'value': len(matched_rows)},
    {'metric': 'names_changed', 'value': len(changed_rows)},
]).to_csv(NAME_UNIT_REMOVED_SUMMARY_CSV, index=False, encoding='utf-8')

print('Done merged cleanup pipeline')
print(f'Source rows: {len(rows)}')
print(f'STEP1 saved: {STEP1_JSON.resolve()} | removed_size_words={size_word_removed} | moved_to_size={size_moved}')
print(f'STEP2 saved: {STEP2_JSON.resolve()} | normalized_to_slice={normalized_to_slice} | removed_multiword={len(removed_multiword)}')
print(f'STEP3 match rows: {len(matched_rows)} | file={STEP3_MATCH_JSON.resolve()}')
print(f'STEP4 final rows: {len(final_rows)} | names_changed={len(changed_rows)} | file={FINAL_JSON.resolve()}')
print('CSV outputs:')
print(f'- {UNIT_COUNTS_AFTER_STEP2_CSV.resolve()}')
print(f'- {MULTIWORD_REMOVED_CSV.resolve()}')
print(f'- {MULTIWORD_SUMMARY_CSV.resolve()}')
print(f'- {OWN_UNIT_MATCH_CSV.resolve()}')
print(f'- {NAME_UNIT_REMOVED_ROWS_CSV.resolve()}')
print(f'- {NAME_UNIT_REMOVED_SUMMARY_CSV.resolve()}')

In [17]:
# Find rows where each row's own unit appears in its name
import json
import re
from pathlib import Path
import pandas as pd

INPUT_JSON = Path('output_filled_copy.json')
OUT_JSON = Path('name_contains_its_own_unit_rows.json')
OUT_CSV = Path('name_contains_its_own_unit_rows.csv')

with INPUT_JSON.open('r', encoding='utf-8') as f:
    rows = json.load(f)

matched = []
for i, r in enumerate(rows):
    unit = str(r.get('unit', '') or '').strip().lower()
    name = str(r.get('name', '') or '').strip().lower()
    if not unit or not name:
        continue

    # Strict token-level match to reduce false positives
    pattern = re.compile(r'\b' + re.escape(unit) + r'\b', re.IGNORECASE)
    if pattern.search(name):
        matched.append({
            'row_index': i,
            'unit': r.get('unit', ''),
            'size': r.get('size', ''),
            'name': r.get('name', ''),
            'gram': r.get('gram', ''),
        })

with OUT_JSON.open('w', encoding='utf-8') as f:
    json.dump(matched, f, ensure_ascii=False, indent=2)

pd.DataFrame(matched).to_csv(OUT_CSV, index=False, encoding='utf-8')

print('Done')
print(f'Total rows: {len(rows)}')
print(f'Rows where name contains its own unit: {len(matched)}')
print('Saved files:')
print(f'- {OUT_JSON.resolve()}')
print(f'- {OUT_CSV.resolve()}')

if matched:
    display(pd.DataFrame(matched).head(100))

Done
Total rows: 18352
Rows where name contains its own unit: 32
Saved files:
- /Users/kyungmin/Desktop/Projects/food_chart_public/food_model/gptgram_model/scrape/name_contains_its_own_unit_rows.json
- /Users/kyungmin/Desktop/Projects/food_chart_public/food_model/gptgram_model/scrape/name_contains_its_own_unit_rows.csv


,row_index,unit,size,name,gram
0,830,leaf,,lettuce leaf,10.00
1,1628,leaf,,green leaf lettuce,10.00
2,4426,leaf,,leaf lettuce,6.00
3,4560,leaf,,fresh basil leaf,0.50
4,4918,leaf,,red leaf lettuce,7.00
5,5107,pod,,green cardamom pod,0.25
6,5119,leaf,,tarragon leaf,0.20
7,6819,drop,,few drop red food coloring,0.05
8,7527,wedge,,lemon wedge,12.00
9,7709,cube,,beef bouillon cube,10.00


In [18]:
# Remove each row's own unit token from name, then save cleaned dataset
import json
import re
from pathlib import Path
import pandas as pd

INPUT_JSON = Path('output_filled_copy.json')
MATCHED_ROWS_JSON = Path('name_contains_its_own_unit_rows.json')
OUTPUT_JSON = Path('output_filled_name_unit_removed.json')
CHANGED_ROWS_CSV = Path('name_unit_removed_rows.csv')
SUMMARY_CSV = Path('name_unit_removed_summary.csv')

with INPUT_JSON.open('r', encoding='utf-8') as f:
    rows = json.load(f)

with MATCHED_ROWS_JSON.open('r', encoding='utf-8') as f:
    matched_rows = json.load(f)

matched_idx = {int(r['row_index']) for r in matched_rows}

changed_rows = []
changed_count = 0

for i, r in enumerate(rows):
    if i not in matched_idx:
        continue

    unit = str(r.get('unit', '') or '').strip()
    name = str(r.get('name', '') or '').strip()
    if not unit or not name:
        continue

    pattern = re.compile(r'\b' + re.escape(unit) + r'\b', re.IGNORECASE)
    cleaned_name = pattern.sub('', name)
    cleaned_name = re.sub(r'\s+', ' ', cleaned_name).strip(' ,.-')

    if cleaned_name != name:
        changed_rows.append({
            'row_index': i,
            'unit': unit,
            'before_name': name,
            'after_name': cleaned_name,
            'size': r.get('size', ''),
            'gram': r.get('gram', ''),
        })
        r['name'] = cleaned_name
        changed_count += 1

with OUTPUT_JSON.open('w', encoding='utf-8') as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

pd.DataFrame(changed_rows).to_csv(CHANGED_ROWS_CSV, index=False, encoding='utf-8')

summary_df = pd.DataFrame([
    {'metric': 'input_rows', 'value': len(rows)},
    {'metric': 'matched_rows_loaded', 'value': len(matched_rows)},
    {'metric': 'names_changed', 'value': changed_count},
])
summary_df.to_csv(SUMMARY_CSV, index=False, encoding='utf-8')

print('Done')
print(f'Input rows: {len(rows)}')
print(f'Matched rows loaded: {len(matched_rows)}')
print(f'Actually changed names: {changed_count}')
print('Saved files:')
print(f'- {OUTPUT_JSON.resolve()}')
print(f'- {CHANGED_ROWS_CSV.resolve()}')
print(f'- {SUMMARY_CSV.resolve()}')

if changed_rows:
    display(pd.DataFrame(changed_rows).head(100))

Done
Input rows: 18352
Matched rows loaded: 32
Actually changed names: 32
Saved files:
- /Users/kyungmin/Desktop/Projects/food_chart_public/food_model/gptgram_model/scrape/output_filled_name_unit_removed.json
- /Users/kyungmin/Desktop/Projects/food_chart_public/food_model/gptgram_model/scrape/name_unit_removed_rows.csv
- /Users/kyungmin/Desktop/Projects/food_chart_public/food_model/gptgram_model/scrape/name_unit_removed_summary.csv


,row_index,unit,before_name,after_name,size,gram
0,830,leaf,lettuce leaf,lettuce,,10.00
1,1628,leaf,green leaf lettuce,green lettuce,,10.00
2,4426,leaf,leaf lettuce,lettuce,,6.00
3,4560,leaf,fresh basil leaf,fresh basil,,0.50
4,4918,leaf,red leaf lettuce,red lettuce,,7.00
5,5107,pod,green cardamom pod,green cardamom,,0.25
6,5119,leaf,tarragon leaf,tarragon,,0.20
7,6819,drop,few drop red food coloring,few red food coloring,,0.05
8,7527,wedge,lemon wedge,lemon,,12.00
9,7709,cube,beef bouillon cube,beef bouillon,,10.00
